## Model Selection and Prompt Optimization

In this notebook, we will demonstrate how the NVIDIA NeMo Agent toolkit (NAT) <a href="https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/evaluate.html">evaluators</a> can be used to develop robust model selection and prompt optimization workflows.

**Goal**: show how the parameter optimizer can be used to compare models and prompts.

The NeMo agent toolkit has extended its low barrier to entry YAML configurable workflow paradigm to support grid searches. (TODO finish description)

## Table of Contents
 
- [Prerequisites](#Prerequisites)
- [API Keys](#API-Keys)
- [Installing NeMo Agent Toolkit](#installing-nemo-agent-toolkit)
 - [Simple Chat Completions Accuracy Comparison with LLM as a Judge](#1-simple-chat-completions-accuracy-comparison-with-llm-as-a-judge)
 - [Head-to-Head Comparison of Multiple LLMs Using Eval](#2-head-to-head-comparison-of-multiple-llms-using-eval)
 - [Model Selection Optimization for Tool Calling Agents](#3-model-selection-optimization-for-tool-calling-agents)
   - [Baseline Tool Calling Invocation](#3a-baseline-tool-calling-invokation)
   - [Preliminary Tool Calling Evaluation Naive Parameters](#3b-preliminary-tool-calling-evaluation-naive-parameters)
   - [Tool Calling Agent Model/Hyperparameter Sweep](#3c-tool-calling-agent-modelhyperparamer-sweep)
   - [Complete Tool Calling Agent Evaluation Optimized](#3d-complete-tool-calling-agent-evaluation-optimized)

### Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

### API Keys

For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

Then you can run the cell below:

In [1]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

## Installing NeMo Agent Toolkit

The recommended way to install NAT is through `pip` or `uv pip`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [2]:
%%bash
pip install uv

NeMo Agent toolkit can be installed through the PyPI `nvidia-nat` package.

There are several optional subpackages available for NAT. For this example, we will rely on two subpackages:
* The `langchain` subpackage contains useful components for integrating and running within [LangChain](https://python.langchain.com/docs/introduction/).
* The `llama-index` subpackage contains useful components for integrating and running within [LlamaIndex](https://developers.llamaindex.ai/python/framework/).

In [ ]:
%%bash
uv pip install "nvidia-nat[langchain,phoenix,profiling]"

## 1) Simple Chat Completions Accuracy Comparison (with LLM as a judge)

Create a basic chat completions workflow (uses LangChain chat completions on backend)

In [ ]:
!nat workflow create tmp --description "A simple chat completions workflow to compare model performance"

Installing workflow 'eval_workflow'...
Workflow 'eval_workflow' installed successfully.
Workflow 'eval_workflow' created successfully in '/Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/notebooks/eval_workflow'.


Let's look at the default configuration of this agent and confirm the agent type, llms, tool calls, and functions...

In [3]:
%%writefile ./eval_workflow/configs/config_a.yml
llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-8b-instruct
    temperature: 0.7
    max_tokens: 1024

workflow:
  _type: chat_completion  # Use the type directly
  system_prompt: |
    You are a helpful AI assistant. Provide clear, accurate, and helpful 
    responses to user queries. Be concise and informative.
  llm_name: nim_llm

Writing ./eval_workflow/configs/config_a.yml


Now let's run this workflow for a simple Q&A example...

In [4]:
!nat run --config_file eval_workflow/configs/config_a.yml --input "Suggest a single name for my new dog"

2025-10-16 19:20:11 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'eval_workflow/configs/config_a.yml'
2025-10-16 19:20:11 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.

Configuration Summary:
--------------------
Workflow Type: chat_completion
Number of Functions: 0
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2025-10-16 19:20:14 - WARNING  - nat.builder.intermediate_step_manager:104 - Step id 0b882c12-9826-484b-a9f6-823e5c9284c8 not found in outstanding start steps
2025-10-16 19:20:14 - INFO     - nat.front_ends.console.console_front

## 2) Head-to-head comparison of multiple LLMs using eval

In this next section, we are going to update the workflow configuration for evaluation and profiling.

Step by step instructions can be found in [4_observability_evaluation_and_profiling.ipynb](./4_observability_evaluation_and_profiling.ipynb). An end to end example of using the Optimizer can be viewed in the [email_phishing_analyzer](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/develop/examples/evaluation_and_profiling/email_phishing_analyzer/src/nat_email_phishing_analyzer/configs/config_optimizer.yml)

The profiler instruments and measures your workflow's performance, while evaluators judge the quality of the outputs. They're separate concepts, so they belong in different sections of the config!

In this next step we will combine the eval and profile configuration into a single config for brevity.

In [6]:
%%writefile eval_workflow/configs/config_b.yml
llms:
  chat_completion_llm:
    _type: nim
    model_name: meta/llama-3.1-8b-instruct
    temperature: 0.0
    max_tokens: 1024
    optimizable_params:
      - model_name
      - temperature
    search_space:
      model_name:
        values:
          - meta/llama-3.1-8b-instruct
          - meta/llama-3.1-70b-instruct
      temperature:
        values:
          - 0.0
          - 0.7

  # Judge LLM for accuracy evaluation
  nim_judge_llm:
    _type: nim
    model_name: meta/llama-3.1-405b-instruct
    temperature: 0.0
    max_tokens: 8  # RAGAS accuracy only needs a score (0-1)

workflow:
  _type: chat_completion
  system_prompt: |
    You are a helpful AI assistant. Provide clear, accurate, and helpful 
    responses to user queries. Be concise and informative.
  llm_name: chat_completion_llm

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: eval_workflow

eval:
  general:
    output_dir: ./eval_workflow/eval_output
    verbose: true
    dataset:
        _type: json
        file_path: ./eval_workflow/data/eval_data.json

  evaluators:
    answer_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_judge_llm
    llm_latency:
      _type: avg_llm_latency
    token_efficiency:
      _type: avg_tokens_per_llm_end

  profiler:
      token_uniqueness_forecast: true
      workflow_runtime_forecast: true
      compute_llm_metrics: true
      csv_exclude_io_text: true
      prompt_caching_prefixes:
        enable: true
        min_frequency: 0.1
      bottleneck_analysis:
        enable_nested_stack: true
      concurrency_spike_analysis:
        enable: true
        spike_threshold: 7

optimizer:
  output_path: ./eval_workflow/eval_output/optimizer/
  reps_per_param_set: 10 # Number of times to evaluate EACH config (for statistical significance)
  eval_metrics:
    accuracy:
      evaluator_name: answer_accuracy  # References the evaluator above
      direction: maximize
    token_efficiency:
      evaluator_name: token_efficiency
      direction: minimize
    latency:
      evaluator_name: llm_latency
      direction: minimize

  numeric:
    enabled: true
    sampler: grid # determines the number of trials to run for each parameter set

  prompt:
    enabled: false  # Disable for pure model comparison

Overwriting eval_workflow/configs/config_b.yml


Adding the test dataset...

In [ ]:
%%writefile eval_workflow/data/eval_data.json
[
    {
        "id": "1",
        "question": "What is 15% of 847?",
        "answer": "127.05"
    },
    {
        "id": "2", 
        "question": "If I invest $10,000 at 5% annual interest compounded monthly for 3 years, how much will I have?",
        "answer": "Approximately $11,614.72"
    },
    {
        "id": "3",
        "question": "What is the current weather in Tokyo?",
        "answer": "This requires real-time weather data for Tokyo, Japan."
    },
    {
        "id": "4",
        "question": "Who won the FIFA World Cup in 2022 and where was it held?",
        "answer": "Argentina won the 2022 FIFA World Cup, which was held in Qatar."
    },
    {
        "id": "5",
        "question": "Calculate the average of these numbers: 23, 45, 67, 89, 12, 34",
        "answer": "The average is 45"
    },
    {
        "id": "6",
        "question": "What is the capital of Australia and what is its approximate population?",
        "answer": "Canberra is the capital of Australia with a population of approximately 460,000 people."
    },
    {
        "id": "7",
        "question": "If a train travels 120 miles in 2 hours, then 180 miles in 3 hours, what is its average speed over the entire journey?",
        "answer": "The average speed is 60 miles per hour (300 miles / 5 hours)."
    },
    {
        "id": "8",
        "question": "Search for information about the latest NASA Mars mission and summarize the key findings.",
        "answer": "Requires web search for current NASA Mars mission information and synthesis of findings."
    },
    {
        "id": "9",
        "question": "What is 2 to the power of 10?",
        "answer": "1024"
    },
    {
        "id": "10",
        "question": "Who is the current CEO of Microsoft and when did they take the position?",
        "answer": "Satya Nadella has been CEO of Microsoft since February 2014."
    },
    {
        "id": "11",
        "question": "Convert 100 degrees Fahrenheit to Celsius and then to Kelvin.",
        "answer": "100°F = 37.78°C = 310.93K"
    },
    {
        "id": "12",
        "question": "Find the top 3 most popular programming languages in 2024 according to recent developer surveys.",
        "answer": "Requires web search for recent programming language popularity surveys from 2024."
    },
    {
        "id": "13",
        "question": "What is the square root of 289?",
        "answer": "17"
    },
    {
        "id": "14",
        "question": "If I start with $1000 and lose 20%, then gain 20% on the new amount, how much do I have?",
        "answer": "$960 (First: $1000 - 20% = $800, Then: $800 + 20% = $960)"
    },
    {
        "id": "15",
        "question": "What are the main differences between Python 3.11 and Python 3.12? Search for official documentation.",
        "answer": "Requires web search for Python 3.12 release notes and feature comparison."
    },
    {
        "id": "16",
        "question": "Calculate: (15 + 25) × 3 - 48 ÷ 6",
        "answer": "112"
    },
    {
        "id": "17",
        "question": "What is the chemical formula for water and what are its key properties?",
        "answer": "H2O. Key properties include: polar molecule, high specific heat capacity, excellent solvent, density maximum at 4°C."
    },
    {
        "id": "18",
        "question": "How many days are there between January 15, 2024 and March 30, 2024?",
        "answer": "75 days"
    },
    {
        "id": "19",
        "question": "Search for the latest NVIDIA GPU announcement and tell me the model name and key specifications.",
        "answer": "Requires web search for recent NVIDIA GPU announcements."
    },
    {
        "id": "20",
        "question": "If a rectangle has a length of 12 cm and a width of 8 cm, what is its area and perimeter?",
        "answer": "Area: 96 cm², Perimeter: 40 cm"
    }
]


Writing eval_workflow/data/eval_data.json


We're also going to start up a local Arize Phoenix server (http://localhost:6006) so that we can view the IO traces for more detailed per-query analysis.

In [ ]:
!uv pip install arize-phoenix

In [8]:
%env PHOENIX_HOST=0.0.0.0

env: PHOENIX_HOST=0.0.0.0


In [10]:
%%bash --bg
# phoenix will run on port 6006
phoenix serve

In [12]:
%%bash
# Optimize across the multi-model search space
nat optimize --config_file eval_workflow/configs/config_b.yml

2025-10-16 19:32:47 - WARNING  - nat.experimental.decorators.experimental_warning_decorator:59 - The Optimizer feature is experimental and the API may change in future releases. Future versions may introduce breaking changes without notice. Function: nat.profiler.parameter_optimization.optimizer_runtime.optimize_config
2025-10-16 19:33:10 - WARNING  - nat.experimental.decorators.experimental_warning_decorator:59 - The Optimizer feature is experimental and the API may change in future releases. Future versions may introduce breaking changes without notice. Function: nat.profiler.parameter_optimization.parameter_optimizer.optimize_parameters
2025-10-16 19:33:10 - INFO     - nat.profiler.parameter_optimization.parameter_optimizer:109 - Grid search enabled: 4 unique parameter combinations to evaluate
[I 2025-10-16 19:33:10,511] A new study created in memory with name: no-name-b1d089a9-f034-4937-b499-cbc2b7f4885d
2025-10-16 19:33:10 - INFO     - nat.profiler.parameter_optimization.parameter

2025-10-16 19:33:14 - INFO     - phoenix.config:1521 - 📋 Ensuring phoenix working directory: /Users/bbednarski/.phoenix
2025-10-16 19:33:14 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_17041ff2-7631-4942-8b28-37c1daa1bd5f initialized
2025-10-16 19:33:24 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.


Running workflow:   0%|          | 0/1 [00:00<?, ?it/s]

2025-10-16 19:33:26 - INFO     - nat.eval.evaluate:446 - Starting evaluation run with config file: general=GeneralConfig(use_uvloop=None, telemetry=TelemetryConfig(logging={'console': ConsoleLoggingMethodConfig(level='INFO')}, tracing={'phoenix': PhoenixTelemetryExporter(project='eval_workflow', endpoint='http://localhost:6006/v1/traces', batch_size=100, flush_interval=5.0, max_queue_size=1000, drop_on_overflow=False, shutdown_timeout=10.0)}), front_end=FastApiFrontEndConfig(root_path='', host='localhost', port=8000, reload=False, workers=1, scheduler_address=None, db_url=None, max_running_async_jobs=10, dask_log_level='WARNING', step_adaptor=StepAdaptorConfig(mode=<StepAdaptorMode.DEFAULT: 'default'>, custom_event_types=[]), workflow=EndpointBase(method='POST', description='Executes the default NAT workflow from the loaded configuration ', path='/generate', websocket_path='/websocket', openai_api_path='/chat', openai_api_v1_path='/v1/chat/completions'), evaluate=EndpointBase(method='P


Running workflow:   0%|          | 0/1 [00:00<?, ?it/s]

...

2025-10-16 19:34:33 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:198 - Parallel coordinates plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_parallel_coordinates.png
2025-10-16 19:34:33 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:198 - Parallel coordinates plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_parallel_coordinates.png
2025-10-16 19:34:33 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:198 - Parallel coordinates plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_parallel_coordinates.png
2025-10-16 19:34:33 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:198 - Parallel coordinates plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_parallel_coordinates.png
2025-10-16 19:34:33 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:198 - Parallel coordinates plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_parallel_c

2025-10-16 19:34:33 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:198 - Parallel coordinates plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_parallel_coordinates.png


2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - na

2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:372 - Visualization complete!
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:374 - Plots saved to: eval_workflow/eval_output/optimizer/plots
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.parameter_optimizer:210 - Pareto visualizations saved to: eval_workflow/eval_output/optimizer/plots
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.optimizer_runtime:66 - All optimization phases complete.


2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - nat.profiler.parameter_optimization.pareto_visualizer:259 - Pairwise matrix plot saved to: eval_workflow/eval_output/optimizer/plots/pareto_pairwise_matrix.png
2025-10-16 19:34:34 - INFO     - na

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast

# Load the optimizer results
trials_df_path = Path("eval_workflow/eval_output/optimizer/trials_dataframe_params.csv")

if trials_df_path.exists():
    trials_df = pd.read_csv(trials_df_path)
    
    print("Grid Search Optimization Results")
    print("=" * 80)
    print("\nTrials Summary:")
    print(trials_df.to_string(index=False))
    
    print("\n" + "=" * 80)
    print("\nModel Performance Statistics (Mean across repetitions):")
    print("-" * 80)
    
    # Group by model name to calculate statistics across repetitions
    for model_name in trials_df['params_llms.chat_completion_llm.model_name'].unique():
        model_trials = trials_df[trials_df['params_llms.chat_completion_llm.model_name'] == model_name]
        
        print(f"\n{model_name}:")
        
        # Parse rep_scores to extract individual repetition metrics
        if 'rep_scores' in model_trials.columns:
            all_accuracies = []
            all_token_efficiencies = []
            all_latencies = []
            
            for rep_scores_str in model_trials['rep_scores']:
                rep_scores = ast.literal_eval(rep_scores_str)
                for score_set in rep_scores:
                    # score_set format: [accuracy, token_efficiency, latency]
                    all_accuracies.append(score_set[0])
                    all_token_efficiencies.append(score_set[1])
                    all_latencies.append(score_set[2])
            
            # Calculate mean and standard deviation
            def calculate_stats(values):
                mean = np.mean(values)
                std = np.std(values)
                ci_lower = np.percentile(values, 2.5)
                ci_upper = np.percentile(values, 97.5)
                return mean, std, ci_lower, ci_upper
            
            acc_mean, acc_std, acc_ci_lower, acc_ci_upper = calculate_stats(all_accuracies)
            tok_mean, tok_std, tok_ci_lower, tok_ci_upper = calculate_stats(all_token_efficiencies)
            lat_mean, lat_std, lat_ci_lower, lat_ci_upper = calculate_stats(all_latencies)
            
            print(f"  Accuracy:")
            print(f"    Mean: {acc_mean:.3f} (±{acc_std:.3f})")
            print(f"    95% CI: [{acc_ci_lower:.3f}, {acc_ci_upper:.3f}]")
            
            print(f"  Token Efficiency:")
            print(f"    Mean: {tok_mean:.3f} (±{tok_std:.3f})")
            print(f"    95% CI: [{tok_ci_lower:.3f}, {tok_ci_upper:.3f}]")
            
            print(f"  Latency:")
            print(f"    Mean: {lat_mean:.3f} (±{lat_std:.3f})")
            print(f"    95% CI: [{lat_ci_lower:.3f}, {lat_ci_upper:.3f}]")
        else:
            # Fallback to aggregated values if rep_scores not available
            # values_0 = accuracy, values_1 = token_efficiency, values_2 = latency
            acc_mean = np.mean(model_trials['values_0'])
            tok_mean = np.mean(model_trials['values_1'])
            lat_mean = np.mean(model_trials['values_2'])
            
            print(f"  Accuracy (mean): {acc_mean:.3f}")
            print(f"  Token Efficiency (mean): {tok_mean:.3f}")
            print(f"  Latency (mean): {lat_mean:.3f}")
            print(f"  Note: 95% CI not available without rep_scores data")
    
    print("\n" + "=" * 80)
    print("\nBest Configuration (by aggregated accuracy across all repetitions):")
    # Find the trial with best aggregated accuracy
    best_trial = trials_df.loc[trials_df['values_0'].idxmax()]
    print(f"Model: {best_trial['params_llms.chat_completion_llm.model_name']}")
    print(f"Temperature: {best_trial['params_llms.chat_completion_llm.temperature']}")
    print(f"Aggregated Accuracy Score: {best_trial['values_0']}")
    print(f"Aggregated Token Efficiency: {best_trial['values_1']}")
    print(f"Aggregated Latency: {best_trial['values_2']}")
else:
    print(f"Optimizer results not found at {trials_df_path}")
    print("Please run the optimizer first (cell 40)")


Grid Search Optimization Results

Trials Summary:
 number  values_0  values_1  values_2             datetime_start          datetime_complete               duration params_llms.chat_completion_llm.model_name  params_llms.chat_completion_llm.temperature                                                                                                                                                                                            rep_scores  system_attrs_grid_id                                                                                                                                  system_attrs_search_space    state
      0     0.800      94.0     1.347 2025-10-16 19:33:10.511983 2025-10-16 19:33:41.852213 0 days 00:00:31.340230                meta/llama-3.1-70b-instruct                                          0.0                [[1.0, 94.0, 1.21], [0.0, 94.0, 1.3], [1.0, 94.0, 1.3], [0.0, 94.0, 1.3], [1.0, 94.0, 1.3], [1.0, 94.0, 1.86], [1.0, 94.0, 1.3], [1.0, 94.0, 1.3]

The results above show:
 
**Grid Search Optimization Summary:**
- The optimizer evaluated all combinations of models and temperatures defined in the search space
- Each configuration was tested multiple times (repetitions) to account for variability
- Three key metrics were tracked: accuracy, token efficiency (tokens used), and latency (response time)
 
 **Understanding the Statistics:**
- **Mean**: Average performance across all repetitions for each model
- **Standard Deviation (±)**: Measure of variability in performance
- **95% Confidence Interval**: Range where we expect 95% of results to fall

**Key Insights:**
 - Different models show different trade-offs between accuracy, efficiency, and speed
- Temperature settings affect response variability and quality
- The "Best Configuration" represents the optimal balance based on the weighted combination of all metrics
 
**Interpreting Your Results:**
When you run this optimization, look for:
- Which model/temperature combination achieves the highest aggregated accuracy
- How token efficiency varies between models (lower is more efficient)
- Latency differences (lower is faster)
- The confidence intervals to understand result stability

The optimizer automatically selects the best configuration and saves it to `optimized_config.yml` for use in production.

## Understanding Evaluation Outputs

This evaluation will have generated two artifacts for analysis at the `output_dir` specified in `config_c.yaml`:
- **trajectory_accuracy_output.json**
- **workflow_output.json**

### Interpreting trajectory_accuracy_output.json

The `trajectory_accuracy_output.json` file contains the results of agent trajectory evaluation.

#### Top-level fields:
- **average_score** - Mean trajectory accuracy score across all evaluated examples (0.0 to 1.0)
- **eval_output_items** - Array of individual evaluation results for each test case

#### Per-item fields:
- **id** - Unique identifier for the test case
- **score** - Trajectory accuracy score for this specific example (0.0 to 1.0)
- **reasoning** - Evaluation reasoning, either:
  - String containing error message if evaluation failed
  - Object with:
    - **reasoning** - LLM judge's explanation of the score
    - **trajectory** - Array of [AgentAction, Output] pairs showing the agent's execution path

The trajectory accuracy evaluator assesses whether the agent used appropriate tools, followed a logical sequence of steps, and efficiently reached the correct answer.

### Interpreting workflow_output.json

The `workflow_output.json` file contains the raw execution results from running the workflow on each test case.

#### Top-level fields:
- **output_items** - Array of workflow execution results for each test case in the dataset

#### Per-item fields:
- **id** - Unique identifier matching the test case ID
- **input_obj** - The input question or prompt sent to the workflow
- **output_obj** - The final answer generated by the workflow
- **trajectory** - Detailed execution trace containing:
  - **event_type** - Type of event (e.g., `LLM_START`, `LLM_END`, `TOOL_START`, `TOOL_END`, `SPAN_START`, `SPAN_END`)
  - **event_timestamp** - Unix timestamp of when the event occurred
  - **metadata** - Event-specific data including:
    - Tool names and inputs
    - LLM prompts and responses
    - Token counts (`prompt_tokens`, `completion_tokens`)
    - Model names
    - Function names
    - Error information

The workflow output provides complete observability into each execution, enabling detailed analysis of agent behavior, performance profiling, and debugging.

## 3) model selection optimization for tool-calling agents

Next, we are going to integrate the [Alert Triage Agent](https://github.com/NVIDIA/NeMo-Agent-Toolkit/tree/develop/examples/advanced_agents/alert_triage_agent) with our optimization pipeline. This agent uses tool calling to automate the triage of server-monitoring alerts. It demonstrates how to build an intelligent troubleshooting workflow using NeMo Agent toolkit and LangGraph.

The Alert Triage Agent is an advanced example that demonstrates:
- **Multi-tool orchestration** - Dynamically selects and uses diagnostic tools
- **Structured report generation** - Creates comprehensive analysis reports
- **Root cause categorization** - Classifies alerts into predefined categories
- **Offline evaluation mode** - Test with synthetic data before live deployment

In this next section, we aim to demonstrate the power of model evaluation and optimization on agentic AI platforms. There are many foundational models to choose as your agent's backbone and academic benchmarks are not always representative of potential performance on your institutionald data (see data leakage [todo: CITE] and domain shift [todo: CITE].

In [14]:
%%bash
# The -e flag installs in "editable" mode, meaning you can modify the source files
# and see changes immediately without reinstalling
uv pip install -e ../../examples/advanced_agents/alert_triage_agent

# This registers all the custom functions (hardware_check, maintenance_check, etc.)
# so they're available to use in our local configuration

Using Python 3.12.11 environment at: /Users/bbednarski/.venvs/unew_312
Resolved 157 packages in 4.85s
   Building nat-alert-triage-agent @ file:///Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/advanced_agents/alert_triage_agent
      Built nat-alert-triage-agent @ file:///Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/advanced_agents/alert_triage_agent
Prepared 1 package in 633ms
Uninstalled 1 package in 12ms
Installed 1 package in 12ms
 ~ nat-alert-triage-agent==1.4.0.dev12+gea18ede6 (from file:///Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/advanced_agents/alert_triage_agent)


### Working Locally with the Alert Triage Agent

The editable installation (`-e`) means:
- The source files remain at `../../examples/advanced_agents/alert_triage_agent`
- You can edit the Python files directly and changes take effect immediately
- All custom functions are registered and available for use
- Your local configuration file (created below) references those functions

**Alternative: Fully Local Workflow**
If you want a completely local workflow without any installation, you could:
1. Create a new workflow: `nat workflow create alert_triage_local`
2. Copy the function implementations to your local `src/` directory
3. Register them in your local `register.py`

For this notebook, we'll use the editable install approach, which gives you the best of both worlds.

### Configuring the Alert Triage Agent

The Alert Triage Agent requires several components:

1. **Diagnostic Tools** - Hardware checks, network connectivity, performance monitoring, telemetry analysis
2. **Sub-agents** - Telemetry metrics analysis agent that coordinates multiple telemetry tools
3. **Categorizer** - Classifies root causes into predefined categories
4. **Maintenance Check** - Filters out alerts during maintenance windows

We'll create a **local configuration file** and run in **offline mode** using synthetic data:


In [29]:
%%writefile ./eval_workflow/configs/alert_triage_config.yml
functions:
  hardware_check:
    _type: hardware_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  host_performance_check:
    _type: host_performance_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  monitoring_process_check:
    _type: monitoring_process_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  network_connectivity_check:
    _type: network_connectivity_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  telemetry_metrics_host_heartbeat_check:
    _type: telemetry_metrics_host_heartbeat_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  telemetry_metrics_host_performance_check:
    _type: telemetry_metrics_host_performance_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  telemetry_metrics_analysis_agent:
    _type: telemetry_metrics_analysis_agent
    tool_names:
      - telemetry_metrics_host_heartbeat_check
      - telemetry_metrics_host_performance_check
    llm_name: agent_llm
  maintenance_check:
    _type: maintenance_check
    llm_name: agent_llm
    static_data_path: ../../examples/advanced_agents/alert_triage_agent/data/maintenance_static_dataset.csv
  categorizer:
    _type: categorizer
    llm_name: agent_llm

workflow:
  _type: alert_triage_agent
  tool_names:
    - hardware_check
    - host_performance_check
    - monitoring_process_check
    - network_connectivity_check
    - telemetry_metrics_analysis_agent
  llm_name: agent_llm
  offline_mode: true
  offline_data_path: ../../examples/advanced_agents/alert_triage_agent/data/offline_data.csv
  benign_fallback_data_path: ../../examples/advanced_agents/alert_triage_agent/data/benign_fallback_offline_data.json

llms:
  agent_llm:
    _type: nim
    model_name: meta/llama-3.1-8b-instruct
    temperature: 0.0
    max_tokens: 2048
    optimizable_params:
      - model_name
      # - temperature
    search_space:
      model_name:
        values:
          - meta/llama-3.1-8b-instruct
          # - meta/llama-3.1-70b-instruct
          # - meta/llama-3.1-403b-instruct
          # - meta/llama-3.3-3b-instruct
          # - meta/llama-3.3-70b-instruct
          # - meta/llama-4-scout-17b-16e-instruct
          # - openai/gpt-oss-20b
          # - openai/gpt-oss-120b
          # - ibm/granite-3.3-8b-instruct
          # - mistralai/mistral-small-3.1-24b-instruct-2503
          # - mistralai/mistral-medium-3-instruct
      # temperature:
      #   values:
      #     - 0.0
      #     - 0.5
      #     - 1.0

  tool_reasoning_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.2
    max_tokens: 2048

  nim_rag_eval_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    max_tokens: 8

eval:
  general:
    output_dir: ./eval_workflow/alert_triage_output/
    dataset:
      _type: json
      file_path: ../../examples/advanced_agents/alert_triage_agent/data/offline_data.json
  evaluators:
    classification_accuracy:
      _type: classification_accuracy
    rag_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_rag_eval_llm

  profiler:
    token_uniqueness_forecast: true
    workflow_runtime_forecast: true
    compute_llm_metrics: true
    csv_exclude_io_text: true
    prompt_caching_prefixes:
      enable: true
      min_frequency: 0.1
    bottleneck_analysis:
      enable_nested_stack: true
    concurrency_spike_analysis:
      enable: true
      spike_threshold: 7

optimizer:
  output_path: ./eval_workflow/alert_triage_output/optimizer/
  reps_per_param_set: 1
  eval_metrics:
    classification_accuracy:
      evaluator_name: classification_accuracy
      direction: maximize
    rag_accuracy:
      evaluator_name: rag_accuracy
      direction: maximize

  numeric:
    enabled: true
    sampler: grid

  prompt:
    enabled: false


Overwriting ./eval_workflow/configs/alert_triage_config.yml


### 3a) Baseline tool calling invokation

Let's test the Alert Triage Agent with a single alert. This alert is an "InstanceDown" alert that, according to the offline dataset, is actually a false positive (the system is healthy).


In [16]:
!nat run --config_file eval_workflow/configs/alert_triage_config.yml \
  --input '{"alert_id": 0, "alert_name": "InstanceDown", "host_id": "test-instance-0.example.com", "severity": "critical", "description": "Instance test-instance-0.example.com is not available for scrapping for the last 5m. Please check: - instance is up and running; - monitoring service is in place and running; - network connectivity is ok", "summary": "Instance test-instance-0.example.com is down", "timestamp": "2025-04-28T05:00:00.000000"}'


2025-10-16 19:41:17 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'eval_workflow/configs/alert_triage_config.yml'
2025-10-16 19:41:18 - INFO     - nat_alert_triage_agent:104 - Preloaded test data from: ../../examples/advanced_agents/alert_triage_agent/data/offline_data.csv
2025-10-16 19:41:18 - INFO     - nat_alert_triage_agent:108 - Preloaded benign fallback data from: ../../examples/advanced_agents/alert_triage_agent/data/benign_fallback_offline_data.json
2025-10-16 19:41:18 - INFO     - nat_alert_triage_agent:80 - ================================================Running in offline mode=================================================

Configuration Summary:
--------------------
Workflow Type: alert_triage_agent
Number of Functions: 9
Number of Function Groups: 0
Number of LLMs: 3
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2025-10-16 19:41:18

After running the cell above, we have confirmed that the tool calling agent is properly configured and ready for a naive evaluation. This evaluation will be our performance baseline.

### 3b) Preliminary tool calling evaluation (naive parameters)
*using "nat eval"...*

Now let's run a full evaluation on the Alert Triage Agent using the complete offline dataset. This dataset contains seven alerts with different root causes:

- **False positives** - System appears healthy despite alert
- **Hardware issues** - Hardware failures or degradation  
- **Software issues** - Malfunctioning monitoring services
- **Maintenance** - Scheduled maintenance windows
- **Repetitive behavior** - Benign recurring patterns

The evaluation will measure:
1. **Classification Accuracy** - How well the agent categorizes root causes
2. **Answer Accuracy** - How well the generated reports match expected outcomes (using RAGAS)


In [17]:
%%bash
nat eval --config_file ./eval_workflow/configs/alert_triage_config.yml


2025-10-16 19:44:20 - INFO     - nat.eval.evaluate:446 - Starting evaluation run with config file: eval_workflow/configs/alert_triage_config.yml
2025-10-16 19:44:43 - INFO     - nat_alert_triage_agent:104 - Preloaded test data from: ../../examples/advanced_agents/alert_triage_agent/data/offline_data.csv
2025-10-16 19:44:43 - INFO     - nat_alert_triage_agent:108 - Preloaded benign fallback data from: ../../examples/advanced_agents/alert_triage_agent/data/benign_fallback_offline_data.json
2025-10-16 19:44:43 - INFO     - nat_alert_triage_agent:80 - ================================================Running in offline mode=================================================
Running workflow:   0%|          | 0/7 [00:00<?, ?it/s]2025-10-16 19:44:45 - INFO     - nat_alert_triage_agent:246 - Host: [test-instance-0.example.com] is NOT under maintenance according to the maintenance database
2025-10-16 19:44:45 - INFO     - nat_alert_triage_agent:246 - Host: [test-instance-1.example.com] is NOT unde

### Understanding Alert Triage Evaluation Results

The evaluation generates several output files in the `alert_triage_output` directory:

1. **classification_accuracy_output.json** - Root cause classification metrics
   - Shows accuracy, precision, recall, and F1 scores for each category
   - Contains confusion matrix for detailed analysis
   
2. **rag_accuracy_output.json** - Answer quality metrics
   - Measures how well generated reports match expected outcomes
   - Uses LLM-as-a-judge to evaluate report quality

3. **workflow_output.json** - Complete execution traces
   - Contains full agent trajectories with tool calls
   - Includes generated reports for each alert
   - Shows token usage and performance metrics

Let's examine the classification accuracy results:


In [18]:
import json

# Load and display classification accuracy results
with open('./eval_workflow/alert_triage_output/classification_accuracy_output.json', 'r') as f:
    classification_results = json.load(f)

print("Classification Accuracy Results:")
print(f"Average Score: {classification_results['average_score']:.2%}")
print("\nPer-Alert Results:")
for item in classification_results['eval_output_items']:
    print(f"  Alert {item['id']}: Score={item['score']:.2f} - {item['reasoning']}")

# Load and display RAG accuracy results
with open('./eval_workflow/alert_triage_output/rag_accuracy_output.json', 'r') as f:
    rag_results = json.load(f)

print("\n\nRAG Accuracy Results:")
print(f"Average Score: {rag_results['average_score']:.2%}")
print(f"Total Alerts Evaluated: {len(rag_results['eval_output_items'])}")


Classification Accuracy Results:
Average Score: 57.00%

Per-Alert Results:
  Alert 0: Score=1.00 - The prediction false_positive is correct. (label: false_positive)
  Alert 1: Score=1.00 - The prediction hardware is correct. (label: hardware)
  Alert 2: Score=1.00 - The prediction software is correct. (label: software)
  Alert 3: Score=0.00 - The prediction ## alert summary is incorrect. (label: maintenance)
  Alert 4: Score=1.00 - The prediction software is correct. (label: software)
  Alert 5: Score=0.00 - The prediction repetitive_behavior is incorrect. (label: false_positive)
  Alert 6: Score=0.00 - The prediction false_positive is incorrect. (label: repetitive_behavior)


RAG Accuracy Results:
Average Score: 50.00%
Total Alerts Evaluated: 7


We see that the classification accuracy results are around 43% based on RAG accuracy results of 46%.

Next we will run the optimizer over a variety of models and some reasonable hyperparameters, then juse that optimal configuration and run the evaluation again.

### 3c) Tool calling agent model/hyperparamer sweep
*using "nat optimize"...*

In [30]:
%%bash
# a multi-model search space
nat optimize --config_file eval_workflow/configs/alert_triage_config.yml

2025-10-16 20:53:39 - WARNING  - nat.experimental.decorators.experimental_warning_decorator:59 - The Optimizer feature is experimental and the API may change in future releases. Future versions may introduce breaking changes without notice. Function: nat.profiler.parameter_optimization.optimizer_runtime.optimize_config
2025-10-16 20:53:44 - WARNING  - nat.experimental.decorators.experimental_warning_decorator:59 - The Optimizer feature is experimental and the API may change in future releases. Future versions may introduce breaking changes without notice. Function: nat.profiler.parameter_optimization.parameter_optimizer.optimize_parameters
2025-10-16 20:53:44 - INFO     - nat.profiler.parameter_optimization.parameter_optimizer:109 - Grid search enabled: 1 unique parameter combinations to evaluate
[I 2025-10-16 20:53:44,065] A new study created in memory with name: no-name-3e72d411-5bbe-422c-973a-cdf4bb3d6fac
2025-10-16 20:53:44 - INFO     - nat.profiler.parameter_optimization.parameter

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast

# Load the optimizer results
trials_df_path = Path("eval_workflow/alert_triage_output/optimizer/trials_dataframe_params.csv")

if trials_df_path.exists():
    trials_df = pd.read_csv(trials_df_path)
    
    print("Grid Search Optimization Results")
    print("=" * 80)
    print("\nTrials Summary:")
    print(trials_df.to_string(index=False))
    
    print("\n" + "=" * 80)
    print("\nModel Performance Statistics (Mean across repetitions):")
    print("-" * 80)
    
    # Group by model name to calculate statistics across repetitions
    for model_name in trials_df['params_llms.agent_llm.model_name'].unique():
        model_trials = trials_df[trials_df['params_llms.agent_llm.model_name'] == model_name]
        
        print(f"\n{model_name}:")
        
        # Parse rep_scores to extract individual repetition metrics
        if 'rep_scores' in model_trials.columns:
            all_classification_accuracies = []
            all_rag_accuracies = []
            
            for rep_scores_str in model_trials['rep_scores']:
                rep_scores = ast.literal_eval(rep_scores_str)
                for score_set in rep_scores:
                    # score_set format: [classification_accuracy, rag_accuracy]
                    all_classification_accuracies.append(score_set[0])
                    all_rag_accuracies.append(score_set[1])
            
            # Calculate mean and standard deviation
            def calculate_stats(values):
                mean = np.mean(values)
                std = np.std(values)
                ci_lower = np.percentile(values, 2.5)
                ci_upper = np.percentile(values, 97.5)
                return mean, std, ci_lower, ci_upper
            
            class_acc_mean, class_acc_std, class_acc_ci_lower, class_acc_ci_upper = calculate_stats(all_classification_accuracies)
            rag_acc_mean, rag_acc_std, rag_acc_ci_lower, rag_acc_ci_upper = calculate_stats(all_rag_accuracies)
            
            print(f"  Classification Accuracy:")
            print(f"    Mean: {class_acc_mean:.3f} (±{class_acc_std:.3f})")
            print(f"    95% CI: [{class_acc_ci_lower:.3f}, {class_acc_ci_upper:.3f}]")
            
            print(f"  RAG Accuracy:")
            print(f"    Mean: {rag_acc_mean:.3f} (±{rag_acc_std:.3f})")
            print(f"    95% CI: [{rag_acc_ci_lower:.3f}, {rag_acc_ci_upper:.3f}]")
        else:
            # Fallback to aggregated values if rep_scores not available
            # values_0 = classification_accuracy, values_1 = rag_accuracy
            class_acc_mean = np.mean(model_trials['values_0'])
            rag_acc_mean = np.mean(model_trials['values_1'])
            
            print(f"  Classification Accuracy (mean): {class_acc_mean:.3f}")
            print(f"  RAG Accuracy (mean): {rag_acc_mean:.3f}")
            print(f"  Note: 95% CI not available without rep_scores data")
    
    print("\n" + "=" * 80)
    print("\nBest Configuration (by aggregated classification accuracy across all repetitions):")
    # Find the trial with best aggregated classification accuracy
    best_trial = trials_df.loc[trials_df['values_0'].idxmax()]
    print(f"Model: {best_trial['params_llms.agent_llm.model_name']}")
    print(f"Aggregated Classification Accuracy Score: {best_trial['values_0']}")
    print(f"Aggregated RAG Accuracy: {best_trial['values_1']}")
else:
    print(f"Optimizer results not found at {trials_df_path}")
    print("Please run the optimizer first (cell 55)")


Grid Search Optimization Results

Trials Summary:
 number  values_0  values_1             datetime_start          datetime_complete               duration params_llms.agent_llm.model_name                   rep_scores  system_attrs_grid_id                                     system_attrs_search_space    state
      0      0.29  0.464286 2025-10-16 20:53:44.065413 2025-10-16 20:55:08.761534 0 days 00:01:24.696121       meta/llama-3.1-8b-instruct [[0.29, 0.4642857142857143]]                     0 {'llms.agent_llm.model_name': ['meta/llama-3.1-8b-instruct']} COMPLETE


Model Performance Statistics (Mean across repetitions):
--------------------------------------------------------------------------------

meta/llama-3.1-8b-instruct:
  Classification Accuracy:
    Mean: 0.290 (±0.000)
    95% CI: [0.290, 0.290]
  RAG Accuracy:
    Mean: 0.464 (±0.000)
    95% CI: [0.464, 0.464]


Best Configuration (by aggregated classification accuracy across all repetitions):
Model: meta/llama-3.1-8b-instr

### 3d) Complete tool calling agent evaluation (optimized)

Now apply the optimal parameters in **optimized_config.yml** to re-evaluate the original eval.

In [34]:
%%bash
nat eval --config_file ./eval_workflow/alert_triage_output/optimizer/optimized_config.yml


2025-10-16 21:00:25 - INFO     - nat.eval.evaluate:446 - Starting evaluation run with config file: eval_workflow/alert_triage_output/optimizer/optimized_config.yml
2025-10-16 21:00:50 - INFO     - nat_alert_triage_agent:104 - Preloaded test data from: ../../examples/advanced_agents/alert_triage_agent/data/offline_data.csv
2025-10-16 21:00:50 - INFO     - nat_alert_triage_agent:108 - Preloaded benign fallback data from: ../../examples/advanced_agents/alert_triage_agent/data/benign_fallback_offline_data.json
2025-10-16 21:00:50 - INFO     - nat_alert_triage_agent:80 - ================================================Running in offline mode=================================================
Running workflow:   0%|          | 0/7 [00:00<?, ?it/s]2025-10-16 21:00:52 - INFO     - nat_alert_triage_agent:246 - Host: [test-instance-0.example.com] is NOT under maintenance according to the maintenance database
2025-10-16 21:00:52 - INFO     - nat_alert_triage_agent:246 - Host: [test-instance-1.examp

In [35]:
import json

# Load and display classification accuracy results
with open('./eval_workflow/alert_triage_output/classification_accuracy_output.json', 'r') as f:
    classification_results = json.load(f)

print("Classification Accuracy Results:")
print(f"Average Score: {classification_results['average_score']:.2%}")
print("\nPer-Alert Results:")
for item in classification_results['eval_output_items']:
    print(f"  Alert {item['id']}: Score={item['score']:.2f} - {item['reasoning']}")

# Load and display RAG accuracy results
with open('./eval_workflow/alert_triage_output/rag_accuracy_output.json', 'r') as f:
    rag_results = json.load(f)

print("\n\nRAG Accuracy Results:")
print(f"Average Score: {rag_results['average_score']:.2%}")
print(f"Total Alerts Evaluated: {len(rag_results['eval_output_items'])}")


Classification Accuracy Results:
Average Score: 71.00%

Per-Alert Results:
  Alert 0: Score=1.00 - The prediction false_positive is correct. (label: false_positive)
  Alert 1: Score=1.00 - The prediction hardware is correct. (label: hardware)
  Alert 2: Score=1.00 - The prediction software is correct. (label: software)
  Alert 3: Score=0.00 - The prediction ## alert summary is incorrect. (label: maintenance)
  Alert 4: Score=0.00 - The prediction repetitive_behavior is incorrect. (label: software)
  Alert 5: Score=1.00 - The prediction false_positive is correct. (label: false_positive)
  Alert 6: Score=1.00 - The prediction repetitive_behavior is correct. (label: repetitive_behavior)


RAG Accuracy Results:
Average Score: 71.43%
Total Alerts Evaluated: 7


Nice to haves:

- Structured report generation (see triage agent)
- model and prompt selection in the same grid search
- question and answer evaluation dataset?
- library of different system prompts for the agent
- automated system prompt generation?